# Install Dependecies

In [1]:
%%capture
%pip install -q "nltk>=3.9,<4" "spacy>=3.8,<4" "transformers>=5,<6"
%pip install matplotlib
!python -m spacy download en_core_web_sm
!python -m spacy download fr_core_news_sm
%pip install ipynbname
%pip install datasets
%pip install ipywidgets

In [2]:
%%capture
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
%pip install ipykernel

# Imports

In [10]:
import ipynbname
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import time
from datasets import load_dataset
from pathlib import Path
from tqdm.std import tqdm
from transformers import AutoTokenizer


# Set `ROOT_DIR`

In [4]:
ROOT_DIR = ipynbname.path().parent
ROOT_DIR = Path(ROOT_DIR)
print(ROOT_DIR)

/home/tlvj/msc_datalogi/2_semester/nlp/msc-nlp-2026/project_notebooks


# Import datasets

In [5]:
from datasets import load_dataset

dataset = load_dataset("coastalcph/tydi_xor_rc")
df_train = dataset["train"].to_pandas()
df_val = dataset["validation"].to_pandas()

### Tokeniser

In [6]:
xlm_tokeniser = AutoTokenizer.from_pretrained("xlm-roberta-base")

# 2 Week 36: Data and Rule-Based Baselines
Download the dataset and inspect its columns. Report Item 1 separately by
language and split, plus overall example counts and answerability proportions.
Compute Item 2 from the training questions separately by language. Apply
Item 3 to every answerable example in both splits.

2. report the five most common question tokens and their counts for each
language, together with an English translation, and explain your tokenisation; and
3. verify programmatically that every answerable item’s answer equals the
substring beginning at answer start, and report the number checked and
any failures.

Implement and evaluate two answerability baselines: (i) the majority-class
baseline estimated from the training split and (ii) a deterministic rule-based
classifier that uses only the question and context. The rule may use tokenisation,
lexical features or machine translation, but no labelled validation examples or
trained answerability/QA model. Discuss what information the rule can and
cannot exploit in this cross-lingual setting.

## 1.1
Report the number of examples, answerable/unanswerable proportions,
median and interquartile range of tokenised question and context lengths
(a table is sufficient; plots are optional), missing values and exact duplicate
question–context pairs

In [7]:
def token_len(texts):
    return [len(xlm_tokeniser(t)["input_ids"]) for t in texts]

In [65]:
df_train["q_len"] = token_len(df_train["question"])
df_train["c_len"] = token_len(df_train["context"])
df_val["q_len"] = token_len(df_val["question"])
df_val["c_len"] = token_len(df_val["context"])

In [57]:
# # merge dataframes
# df_train["split"] = "train"
# df_val["split"] = "validation"

# df = pd.concat([df_train, df_val])
# df.head()

In [66]:
# compute the number of answerable / unanswerable / pct_unanswerable
counts = df_train.groupby(["split", "lang"])["answerable"].agg(n="count", answerable="sum")
counts["unanswerable"] = counts["n"] - counts["answerable"]
counts["pct_unanswerable"] = (100 * counts["unanswerable"] / counts["n"]).round(2)
print(counts.to_markdown())

|                 |    n |   answerable |   unanswerable |   pct_unanswerable |
|:----------------|-----:|-------------:|---------------:|-------------------:|
| ('train', 'ar') | 2558 |         2303 |            255 |               9.97 |
| ('train', 'bn') | 2598 |         2449 |            149 |               5.74 |
| ('train', 'fi') | 2126 |         1863 |            263 |              12.37 |
| ('train', 'ja') | 2301 |         1928 |            373 |              16.21 |
| ('train', 'ko') | 2422 |         2359 |             63 |               2.6  |
| ('train', 'ru') | 1983 |         1750 |            233 |              11.75 |
| ('train', 'te') | 1355 |         1310 |             45 |               3.32 |


In [67]:
# compute the number of answerable / unanswerable / pct_unanswerable
counts = df_val.groupby(["split", "lang"])["answerable"].agg(n="count", answerable="sum")
counts["unanswerable"] = counts["n"] - counts["answerable"]
counts["pct_unanswerable"] = (100 * counts["unanswerable"] / counts["n"]).round(2)
print(counts.to_markdown())

|                      |   n |   answerable |   unanswerable |   pct_unanswerable |
|:---------------------|----:|-------------:|---------------:|-------------------:|
| ('validation', 'ar') | 415 |          363 |             52 |              12.53 |
| ('validation', 'bn') | 476 |          371 |            105 |              22.06 |
| ('validation', 'fi') | 528 |          380 |            148 |              28.03 |
| ('validation', 'ja') | 456 |          287 |            169 |              37.06 |
| ('validation', 'ko') | 356 |          337 |             19 |               5.34 |
| ('validation', 'ru') | 396 |          284 |            112 |              28.28 |
| ('validation', 'te') | 384 |          291 |             93 |              24.22 |


In [68]:
ratio = df_train.groupby("split")["answerable"].agg(n="count", answerable="sum")
ratio["unanswerable"] = ratio["n"] - ratio["answerable"]
ratio["pct_unanswerable"] = (100 * ratio["unanswerable"] / ratio["n"]).round(1)
print(ratio.to_markdown())

| split   |     n |   answerable |   unanswerable |   pct_unanswerable |
|:--------|------:|-------------:|---------------:|-------------------:|
| train   | 15343 |        13962 |           1381 |                  9 |


In [69]:
ratio = df_val.groupby("split")["answerable"].agg(n="count", answerable="sum")
ratio["unanswerable"] = ratio["n"] - ratio["answerable"]
ratio["pct_unanswerable"] = (100 * ratio["unanswerable"] / ratio["n"]).round(1)
print(ratio.to_markdown())

| split      |    n |   answerable |   unanswerable |   pct_unanswerable |
|:-----------|-----:|-------------:|---------------:|-------------------:|
| validation | 3011 |         2313 |            698 |               23.2 |


In [70]:
for col in ["q_len", "c_len"]:
    grouped = df_train.groupby(["split", "lang"])[col]
    table = pd.DataFrame({"median": grouped.median(), "q25": grouped.quantile(0.25), "q75": grouped.quantile(0.75)})
    table["iqr"] = table["q75"] - table["q25"]
    print(table.to_markdown())

|                 |   median |   q25 |   q75 |   iqr |
|:----------------|---------:|------:|------:|------:|
| ('train', 'ar') |       13 |    11 |    15 |     4 |
| ('train', 'bn') |       17 |    14 |    21 |     7 |
| ('train', 'fi') |       12 |    10 |    14 |     4 |
| ('train', 'ja') |       14 |    12 |    17 |     5 |
| ('train', 'ko') |       14 |    12 |    16 |     4 |
| ('train', 'ru') |       15 |    12 |    18 |     6 |
| ('train', 'te') |       13 |    11 |    16 |     5 |
|                 |   median |   q25 |   q75 |   iqr |
|:----------------|---------:|------:|------:|------:|
| ('train', 'ar') |      133 |  89   | 191   |   102 |
| ('train', 'bn') |      130 |  87   | 186   |    99 |
| ('train', 'fi') |      132 |  90   | 191   |   101 |
| ('train', 'ja') |      132 |  88   | 188   |   100 |
| ('train', 'ko') |      127 |  85   | 181   |    96 |
| ('train', 'ru') |      137 |  88.5 | 194.5 |   106 |
| ('train', 'te') |      121 |  80   | 171   |    91 |


In [71]:
for col in ["q_len", "c_len"]:
    grouped = df_val.groupby(["split", "lang"])[col]
    table = pd.DataFrame({"median": grouped.median(), "q25": grouped.quantile(0.25), "q75": grouped.quantile(0.75)})
    table["iqr"] = table["q75"] - table["q25"]
    print(table.to_markdown())

|                      |   median |   q25 |   q75 |   iqr |
|:---------------------|---------:|------:|------:|------:|
| ('validation', 'ar') |       12 |    11 |    15 |     4 |
| ('validation', 'bn') |       17 |    15 |    21 |     6 |
| ('validation', 'fi') |       12 |    11 |    15 |     4 |
| ('validation', 'ja') |       14 |    12 |    17 |     5 |
| ('validation', 'ko') |       14 |    12 |    16 |     4 |
| ('validation', 'ru') |       14 |    12 |    17 |     5 |
| ('validation', 'te') |       14 |    12 |    18 |     6 |
|                      |   median |    q25 |    q75 |    iqr |
|:---------------------|---------:|-------:|-------:|-------:|
| ('validation', 'ar') |    131   |  89.5  | 190.5  | 101    |
| ('validation', 'bn') |    140   |  90    | 180.25 |  90.25 |
| ('validation', 'fi') |    149   | 104    | 207    | 103    |
| ('validation', 'ja') |    120   |  93.75 | 166    |  72.25 |
| ('validation', 'ko') |    123.5 |  80.75 | 183.5  | 102.75 |
| ('validation', 'r

In [33]:
print(df.isna().sum())
print("----------------------")
print((df == "").sum())

question             0
context              0
lang                 0
answerable           0
answer_start         0
answer               0
answer_inlang    17604
q_len                0
c_len                0
split                0
dtype: int64
----------------------
question         0
context          0
lang             0
answerable       0
answer_start     0
answer           0
answer_inlang    0
q_len            0
c_len            0
split            0
dtype: int64


In [40]:
print(df[df["answerable"]]["answer_start"].unique())
print(df.groupby("lang")["answer_inlang"].count())

[ 182   48   39 ... 1070 2023  636]
lang
ar      0
bn    150
fi    150
ja    150
ko      0
ru    150
te    150
Name: answer_inlang, dtype: int64


In [41]:
print("train:", df_train.duplicated(subset=["question", "context"]).sum())
print("validation:", df_val.duplicated(subset=["question", "context"]).sum())

train: 17
validation: 0


In [42]:
print(df[df.duplicated(subset=["split", "question", "context"], keep=False)].groupby(["split", "lang"]).size())

split  lang
train  bn      14
       ko      20
dtype: int64
